# Solving the Poisson Equation with TensorMesh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson.ipynb)

The "hello world" of finite elements, on TensorMesh's full pipeline:
**Mesh → Assembler → SparseMatrix → Condenser → Solve** — all in PyTorch.

We solve $-\Delta u = f$ on the unit square with homogeneous Dirichlet
boundary conditions, using a manufactured multi-frequency solution so the
numerical answer can be checked against the exact one.

Docs: [Poisson Equation — Basic 2D](https://docs.tensor-mesh.com/example_gallery/poisson.html#basic-2d-poisson-poisson-py) · Source: [`examples/poisson/poisson.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/poisson/poisson.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Mesh and stiffness matrix

`Mesh.gen_rectangle` calls gmsh under the hood and returns a triangulated
unit square. Subclassing would let us write any weak form; here the built-in
`LaplaceElementAssembler` provides $\int_\Omega \nabla u \cdot \nabla v \, dx$.

In [ ]:
import torch
from tensormesh import LaplaceElementAssembler, Mesh, Condenser, NodeAssembler
from tensormesh.dataset import PoissonMultiFrequency

with quiet():
    mesh = Mesh.gen_rectangle(chara_length=0.02).double()
print(f"mesh: {mesh.n_points} nodes")

assembler = LaplaceElementAssembler.from_mesh(mesh)
K = assembler(mesh.points)
print(f"stiffness matrix: {K.shape}, {K.values.shape[0]} non-zeros")

## Right-hand side

A `NodeAssembler` integrates the load: its `forward` is the integrand
$f \, v$, and extra fields (here the source $f$ sampled at the nodes) are
passed via `point_data`.

In [ ]:
class FAssembler(NodeAssembler):
    def forward(self, v, f):
        return v * f

equation = PoissonMultiFrequency(K=16)
f = equation.source_term(mesh.points, domain="rectangle")
b = FAssembler.from_mesh(mesh)(mesh.points, point_data={"f": f})

## Boundary conditions and solve

`Condenser` applies Dirichlet conditions by static condensation: it splits
the system into free and prescribed DOFs, solves the inner system, and
`recover` re-inserts the boundary values.

In [ ]:
condenser = Condenser(mesh.boundary_mask, torch.zeros(mesh.n_points, dtype=torch.float64))
K_, b_ = condenser(K, b)
u = condenser.recover(K_.solve(b_))

u_exact = equation.solution(mesh.points)
rel_l2 = torch.norm(u - u_exact) / torch.norm(u_exact)
print(f"relative L2 error vs. analytical solution: {rel_l2:.3e}")

In [ ]:
mesh.plot({"f": f, "u_fem": u, "u_analytical": u_exact}, save_path="poisson.png")
import matplotlib.pyplot as plt
plt.close("all")  # mesh.plot leaves its figure open; avoid a duplicate inline render
from IPython.display import Image
Image("poisson.png")

## Where to next

- Everything above is differentiable: wrap the solve in an optimization loop
  and backpropagate through the FEM solution — see the
  [coefficient identification notebook](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/coefficient_identification.ipynb).
- GPU solving: `pip install "tensormesh-fem[gpu]"` and move the mesh with
  `.to("cuda")` — the sparse solve dispatches to cuDSS.
- More examples: [the full gallery](https://docs.tensor-mesh.com/example_gallery/index.html).